In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

/home/rikesh/Rikesh/RAG/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [13]:
load_dotenv()

True

In [3]:
# temperature=0 ensures deterministic entity and relationship extraction
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("data/elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

Page 1: 2343 chars
Page 2: 1192 chars


In [5]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [8]:
print(os.getenv("NEO4J_URI"))

neo4j+s://e16be8d0.databases.neo4j.io


In [10]:
print(os.getenv("NEO4J_USERNAME"))
print(os.getenv("NEO4J_DATABASE"))

e16be8d0
None


In [14]:
print(os.getenv("NEO4J_DATABASE"))

e16be8d0


In [22]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"],
    timeout=120,
)

In [16]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [23]:
print(graph.query("RETURN 1 AS result"))

[{'result': 1}]


In [17]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

14 graph documents extracted
Nodes: ['Elon Musk', 'June 28, 1971', 'Pretoria, South Africa', 'American', 'Entrepreneur', 'Engineer', "World'S Wealthiest Person"]
Rels:  [('Elon Musk', 'BORN_ON', 'June 28, 1971'), ('Elon Musk', 'BORN_IN', 'Pretoria, South Africa'), ('Elon Musk', 'HAS_NATIONALITY', 'American'), ('Elon Musk', 'HAS_OCCUPATION', 'Entrepreneur'), ('Elon Musk', 'HAS_OCCUPATION', 'Engineer'), ('Elon Musk', 'HAS_RECOGNITION', "World'S Wealthiest Person")]


In [24]:
print("Number of graph documents:", len(graph_docs))

total_nodes = sum(len(doc.nodes) for doc in graph_docs)
total_relationships = sum(len(doc.relationships) for doc in graph_docs)

print("Total nodes:", total_nodes)
print("Total relationships:", total_relationships)

Number of graph documents: 14
Total nodes: 113
Total relationships: 120


In [19]:
from neo4j import GraphDatabase
import os

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(
        os.environ["NEO4J_USERNAME"],
        os.environ["NEO4J_PASSWORD"],
    ),
)

driver.verify_connectivity()

print("Neo4j connection successful!")

Neo4j connection successful!


In [20]:
with driver.session(database=os.environ["NEO4J_DATABASE"]) as session:
    result = session.run("RETURN 1 AS result")
    print(result.single())

<Record result=1>


In [21]:
with driver.session(database=os.environ["NEO4J_DATABASE"]) as session:
    session.run("""
        CREATE CONSTRAINT IF NOT EXISTS
        FOR (b:__Entity__)
        REQUIRE b.id IS UNIQUE
    """).consume()

print("Constraint created successfully")

Constraint created successfully


In [26]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell

test_docs = graph_docs[:1]

print("Nodes:", len(test_docs[0].nodes))
print("Relationships:", len(test_docs[0].relationships))

graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

Nodes: 7
Relationships: 6
Graph stored in Neo4J


In [28]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [29]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['__Entity__', 'Person'], 'count': 23}
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Organization'], 'count': 11}
  {'label': ['__Entity__', 'Location'], 'count': 11}
  {'label': ['__Entity__', 'Product'], 'count': 11}
  {'label': ['__Entity__', 'Date'], 'count': 6}
  {'label': ['__Entity__', 'Occupation'], 'count': 5}
  {'label': ['__Entity__', 'Concept'], 'count': 3}
  {'label': ['__Entity__', 'Country'], 'count': 2}
  {'label': ['__Entity__', 'Company'], 'count': 2}
  {'label': ['__Entity__', 'Year'], 'count': 2}
  {'label': ['__Entity__', 'Service'], 'count': 2}
  {'label': ['__Entity__', 'Activity'], 'count': 2}
  {'label': ['__Entity__', 'Place'], 'count': 1}
  {'label': ['__Entity__', 'Nationality'], 'count': 1}
  {'label': ['__Entity__', 'Recognition'], 'count': 1}
  {'label': ['__Entity__', 'City'], 'count': 1}
  {'label': ['__Entity__', 'Country', 'Location'], 'count': 1}
  {'label': ['__Entity__', 'Amount'], 'count': 1}
Relationships:
